# WanGP Character Consistency - RunPod / Jupyter

Run this notebook from top to bottom. The only cells you normally edit are **Cell 2** for paths/settings and **Cell 8** for your character/reference details.

Recommended RunPod template: a PyTorch CUDA image with Jupyter and at least 24 GB VRAM for the first Bernini/VACE tests. Smaller GPUs can try `bernini_ingredients_1_3b`.

## Cell 1 - Check GPU

In [ ]:
!nvidia-smi

## Cell 2 - Edit Your Basic Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/SohailAwg/wan2gp_studio.git"
ROOT = Path("/workspace/Wan2GP-main")
REF_DIR = Path("/workspace/refs")
OUTPUT_DIR = Path("/workspace/wan2gp_outputs")

# Set this True only on a fresh pod. After dependencies are installed once, set False to save time.
INSTALL_DEPS = True

# Safe first-run attention. Later you can try "sage2" after SageAttention is installed.
ATTENTION = "sdpa"

# WanGP memory profile. 4 is a practical default for RunPod GPUs; reduce/increase only if needed.
PROFILE = "4"

ROOT, REF_DIR, OUTPUT_DIR

## Cell 3 - Clone Or Update The Repo

In [ ]:
import os
import subprocess
from pathlib import Path

if ROOT.exists():
    print(f"Repo exists: {ROOT}")
    os.chdir(ROOT)
    subprocess.run(["git", "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
else:
    os.chdir("/workspace")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)

REF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready:", ROOT)

## Cell 4 - Install Python Dependencies

Run this on a fresh pod. If your RunPod image already has matching PyTorch, this still checks the environment first.

In [ ]:
import importlib.util
import subprocess
import sys

if INSTALL_DEPS:
    os.chdir(ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    if importlib.util.find_spec("torch") is None:
        subprocess.run([
            sys.executable, "-m", "pip", "install",
            "torch==2.10.0", "torchvision==0.25.0", "torchaudio==2.10.0",
            "--index-url", "https://download.pytorch.org/whl/cu130",
        ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Skipping dependency install because INSTALL_DEPS=False")

## Cell 5 - Load Secrets Safely

Best format for `/workspace/secrets.txt`:

```text
HF_TOKEN=hf_...
RUNPOD_API_KEY=...
```

This cell does not print secret values.

In [ ]:
import os
from pathlib import Path

secret_candidates = [Path('/workspace/secrets.txt'), ROOT / 'secrets.txt']

def load_secret_file(path: Path):
    if not path.exists():
        return []
    loaded = []
    raw_lines = []
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if '=' in line:
            key, value = line.split('=', 1)
        elif ':' in line:
            key, value = line.split(':', 1)
        else:
            raw_lines.append(line)
            continue
        key, value = key.strip(), value.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded.append(key)
    for value in raw_lines:
        if value.startswith('hf_'):
            os.environ['HF_TOKEN'] = value
            loaded.append('HF_TOKEN')
        else:
            # Raw RunPod keys are not needed inside this notebook unless you create pods from code.
            os.environ.setdefault('RUNPOD_API_KEY', value)
            loaded.append('RUNPOD_API_KEY')
    return loaded

loaded = []
for path in secret_candidates:
    loaded += load_secret_file(path)

print('Loaded secret names:', sorted(set(loaded)) if loaded else 'none')
print('HF_TOKEN available:', bool(os.environ.get('HF_TOKEN')))
print('RUNPOD_API_KEY available:', bool(os.environ.get('RUNPOD_API_KEY')))

## Cell 6 - Optional Hugging Face Login

This helps gated Hugging Face models download correctly.

In [ ]:
if os.environ.get('HF_TOKEN'):
    !huggingface-cli login --token $HF_TOKEN --add-to-git-credential
else:
    print('No HF_TOKEN found. Public models may still download, but gated models can fail.')

## Cell 7 - Put Reference Images In Place

Upload your character images to `/workspace/refs` from the Jupyter file browser. Use clear names like:

- `character_front.png`
- `character_body.png`
- `character_profile.png`

Then run this cell to list what Jupyter can see.

In [ ]:
REF_DIR.mkdir(parents=True, exist_ok=True)
for path in sorted(REF_DIR.glob('*')):
    print(path)

## Cell 8 - Edit Character And Shot Details

For 24 GB+ VRAM, start with `bernini_ingredients`. For lower VRAM, try `bernini_ingredients_1_3b`.

In [ ]:
CHARACTER_NAME = "my_character"

# Modes:
# - bernini_ingredients: best multi-reference ingredients workflow, heavier
# - bernini_ingredients_1_3b: lower VRAM, lower quality
# - vace_standin: identity + control workflow
# - standin_face: one close-up face reference
# - scail2_animate: reference character + motion/control video
# - joyai_echo_memory: multi-shot memory story, no image refs by default
MODE = "bernini_ingredients"

REFS = [
    str(REF_DIR / "character_front.png"),
    str(REF_DIR / "character_body.png"),
    str(REF_DIR / "character_profile.png"),
]

IDENTITY_PROMPT = """
A consistent named character. Replace this with exact face shape, eye color, hairstyle, age, body proportions,
signature clothing, color palette, accessories, and persistent props. The same person must appear in every shot.
""".strip()

ENVIRONMENT_PROMPT = """
A recurring world. Replace this with the location, era, architecture, props, weather, lighting direction,
background details, and color palette that should remain stable across shots.
""".strip()

STYLE_PROMPT = "cinematic natural light, realistic texture, stable color grade"

NEGATIVE_PROMPT = "different person, face drift, changed hairstyle, changed age, inconsistent outfit, changed location, inconsistent environment, distorted hands, extra fingers"

SHOT_PROMPTS = [
    "Medium close-up, the character looks toward camera and smiles subtly, gentle camera push-in.",
    "Wide shot, the same character walks through a quiet city street at dusk, same outfit and silhouette.",
]

RESOLUTION = "1280x720"
VIDEO_LENGTH = 121
SEED = -1

# Only needed for SCAIL/VACE control workflows.
CONTROL_VIDEO = ""

missing = [path for path in REFS if MODE != 'joyai_echo_memory' and not Path(path).exists()]
print('Missing reference files:', missing if missing else 'none')

## Cell 9 - Build WanGP Settings Manifest

In [ ]:
import importlib.util
import json

helper_path = ROOT / "plugins" / "wan2gp-character-consistency" / "character_pack.py"
spec = importlib.util.spec_from_file_location("wan2gp_character_pack", helper_path)
packs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(packs)

manifest = packs.build_manifest(
    character_name=CHARACTER_NAME,
    mode=MODE,
    identity_prompt=IDENTITY_PROMPT,
    environment_prompt=ENVIRONMENT_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    style_prompt=STYLE_PROMPT,
    shot_prompts=SHOT_PROMPTS,
    refs=REFS,
    resolution=RESOLUTION,
    video_length=VIDEO_LENGTH,
    seed=SEED,
    control_video=CONTROL_VIDEO,
)

settings_path = packs.save_manifest_json(CHARACTER_NAME, manifest)
print("Settings saved to:", settings_path)
print(json.dumps(manifest[0], indent=2)[:3000])

## Cell 10 - Start WanGP API Session

This loads WanGP in-process. First run may download model files when generation begins.

In [ ]:
import os
import sys
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from shared.api import init

session = init(
    root=ROOT,
    output_dir=OUTPUT_DIR,
    cli_args=["--attention", ATTENTION, "--profile", PROFILE],
)

print("WanGP API session ready")

## Cell 11 - Generate The First Shot Only

Use this for a quick smoke test before running all shots.

In [ ]:
first_settings = manifest[0]
job = session.submit_task(first_settings)
result = job.result()

print("success:", result.success)
print("files:", result.generated_files)
print("errors:", [str(error) for error in result.errors])

## Cell 12 - Preview The First Output

In [ ]:
from IPython.display import Video, display

if result.success and result.generated_files:
    output_path = result.generated_files[0]
    print(output_path)
    display(Video(output_path, embed=True, width=720))
else:
    print("No output video to preview.")

## Cell 13 - Generate All Shots

Run this only after Cell 11 looks good.

In [ ]:
all_job = session.submit(settings_path)
all_result = all_job.result()

print("success:", all_result.success)
print("files:")
for file in all_result.generated_files:
    print(file)
print("errors:", [str(error) for error in all_result.errors])

## Cell 14 - Troubleshooting Quick Fixes

- Out of memory: change `MODE` to `bernini_ingredients_1_3b`, reduce `RESOLUTION` to `832x480`, or reduce `VIDEO_LENGTH` to `81`.
- Hugging Face gated-model error: make sure `HF_TOKEN=...` is in `/workspace/secrets.txt`, rerun Cells 5-6, and confirm your HF account accepted the model terms.
- Missing references: upload images into `/workspace/refs` and rerun Cells 7-9.
- Slow generation: first run downloads models. Later runs are faster if the pod storage persists.
- Better speed later: install SageAttention, then change `ATTENTION = "sage2"` in Cell 2.